# Course Classification and Credits

This notebook adds a `course_type` column with values like `theory`, `lab`, `embedded`, and `project`, then extracts `total_credits` from `courses.pdf` and exports an enriched CSV.

In [1]:
import csv
import re
from pathlib import Path

from pypdf import PdfReader

base = Path('.').resolve()
source_csv = base / 'data' / 'courses.csv'
pdf_path = base / 'data' / 'courses.pdf'
out_csv = base / 'data' / 'courses_enriched.csv'

with source_csv.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

reader = PdfReader(str(pdf_path))
pdf_text = '\n'.join(page.extract_text() or '' for page in reader.pages)
pdf_text = pdf_text.replace('\r', '\n')
pdf_text = re.sub(r'[ \t]+', ' ', pdf_text)
pdf_text = re.sub(r'\n{2,}', '\n', pdf_text)

suffix_map = {
    'L': 'theory',
    'P': 'lab',
    'E': 'embedded',
    'J': 'project',
    'M': 'theory',
}

section_starters = [
    'Foundation Core',
    'Discipline-linked Engineering Sciences',
    'Discipline Core',
    'Projects and Internship',
    'Open Elective',
    'Non-graded Core Requirement',
    'Bridge Course',
    'Specialization Elective',
]
section_regex = '|'.join(re.escape(x) for x in section_starters)


def classify_course(code: str) -> str:
    return suffix_map.get(code[-1].upper(), 'other')


def extract_total_credits(code: str):
    match = re.search(
        rf'{re.escape(code)}\s+(.*?)(?=\n\d+\s+[A-Z]{{4}}\d{{3}}[A-Z]?|\n(?:{section_regex})\b|\Z)',
        pdf_text,
        re.S,
    )
    if not match:
        return None
    chunk = re.sub(r'\s+', ' ', match.group(1)).strip()
    numbers = re.findall(r'(\d+\.\d+)', chunk)
    if not numbers:
        return None
    return float(numbers[-1])


output_rows = []
for row in rows:
    code = row['course_id'].strip()
    output_rows.append({
        'course_id': code,
        'course_name': row['course_name'].strip(),
        'type': classify_course(code),
        'credits': extract_total_credits(code),
    })

with out_csv.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['course_id', 'course_name', 'type', 'credits'])
    writer.writeheader()
    writer.writerows(output_rows)

print(f'Wrote {len(output_rows)} rows to {out_csv}')
print('Sample rows:')
for row in output_rows[:10]:
    print(row)
print('\nType counts:')
counts = {}
for row in output_rows:
    counts[row['type']] = counts.get(row['type'], 0) + 1
print(counts)
print('\nMissing credits:', sum(1 for row in output_rows if row['credits'] is None))

Wrote 695 rows to C:\Users\aditya\Desktop\Project\facReviewWEBSITE\data\courses_enriched.csv
Sample rows:
{'course_id': 'BCHY101L', 'course_name': 'Engineering Chemistry', 'type': 'theory', 'credits': 3.0}
{'course_id': 'BCHY101P', 'course_name': 'Engineering Chemistry Lab', 'type': 'lab', 'credits': 1.0}
{'course_id': 'BMAT101L', 'course_name': 'Calculus', 'type': 'theory', 'credits': 3.0}
{'course_id': 'BMAT101P', 'course_name': 'Calculus Lab', 'type': 'lab', 'credits': 1.0}
{'course_id': 'BMAT102L', 'course_name': 'Differential Equations and Transforms', 'type': 'theory', 'credits': 4.0}
{'course_id': 'BMAT201L', 'course_name': 'Complex Variables and Linear Algebra', 'type': 'theory', 'credits': 4.0}
{'course_id': 'BMAT202L', 'course_name': 'Probability and Statistics', 'type': 'theory', 'credits': 3.0}
{'course_id': 'BMAT202P', 'course_name': 'Probability and Statistics Lab', 'type': 'lab', 'credits': 1.0}
{'course_id': 'BPHY101L', 'course_name': 'Engineering Physics', 'type': 'the